# 50. Is the low rank caused by the hypernetwork, or by the measure?

## Why this exists

Notebook 48 found that the effective rank this study reports is almost all one
direction. That direction is the average feature row, the part every picture
has in common. About 98 percent of the feature matrix is that one row. Take it
out and effective rank goes from about 2 to about 9.

That is the strongest finding in the study, and it has one obvious objection.
A reviewer will say: your model is a hypernetwork. A generator writes its
weights from a 32 number code, which is an unusual way to build a network, so
of course its features are degenerate. Your result is about your architecture,
not about the measure.

This notebook answers that objection. If an ordinary network, with ordinary
weights, read on the same pictures at the same place, also comes out about 98
percent average row, then the finding is about the measure. It would then
apply to anyone who reports effective rank on ReLU features, which is most of
the plasticity literature.

Nothing is trained here. Every reading is a forward pass.

## The four networks

Each one isolates a different possible cause.

| Network | What it rules in or out |
| --- | --- |
| 1. The study's own target, ordinary random weights | Same architecture, same pictures, but the weights come from normal initialisation instead of the generator. If this reads 0.98 too, the generator is not the cause. |
| 2. Stock ResNet50, random weights | Removes the study's changed stem as a cause. |
| 3. Stock ResNet50, trained on ImageNet | A strongly trained ordinary network. Rules out "only untrained networks do this". |
| 4. Stock ResNet50, trained on ImageNet, at its own 224 size with its own normalisation | Rules out "you fed it the wrong size and the wrong scaling". |

Network 1 is the sharpest test. It differs from the study's models in exactly
one way: where the weights come from.

## What the answer will mean

**If all four read a mean share near 0.98**, the finding is about the measure
and not about this study. That is a bigger result than the one you set out to
find, and it needs no more GPU time.

**If only the hypernetwork reads 0.98** and ordinary networks read much lower,
the finding is about hypernetworks. Still publishable, but narrower, and the
paper has to say so.

**If the ordinary networks read higher but still above about 0.9**, the truth
is in between. Report both numbers and let the reader judge.

## 1. Setup

This needs no GPU worth speaking of. It pushes 256 pictures through four
networks. A CPU runtime will finish it, slower on the 224 size network. Use a
GPU only if one is free, because the probe run may be using it.

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules

# Importing uncle twice in one session gives two copies of the module with two
# copies of its constants, which is a confusing way to fail much later.
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")

if IN_COLAB:
    REPO = Path("/content/path-indicators")
    if not REPO.exists():
        subprocess.run(["git", "clone",
                        "https://github.com/sumitasthana/PATH.git", str(REPO)],
                       check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "uncle/task_partition_10x20.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this from a checkout that has the ten-group split.")

for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None and IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package],
                       check=True)
sys.path.insert(0, str(REPO))

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50, ResNet50_Weights

from uncle.config import Config, PROBE_TASK, study_defaults
from uncle.hypernet import build_target
from uncle.plasticity_indicators import SRANK_DELTA, fixed_image_sets
from uncle.storage import R2Store
from uncle.streams import build_tasks
from uncle.tasks import PARTITIONS

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                               text=True).strip()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running at:", HEAD)
print("Device:", DEVICE)

## 2. Settings

Everything here is copied from the study so the readings are comparable. The
seed that picks the pictures, the number of pictures, and the srank cut-off all
come from the library rather than being retyped, so they cannot drift apart.

In [ ]:
STUDY_ID = "plasticity_revised_v1"
BUCKET = "uncle-experiments"

# Where this notebook's readings are saved. A separate key from notebook 48's,
# because these are different models and mixing them in one file would be a
# quiet way to confuse two protocols.
SAVE_KEY = f"uncle/indicator_protocol/{STUDY_ID}/plain_network_centering.jsonl"

# Notebook 48's readings, for the side by side table at the end.
HYPERNET_KEY = f"uncle/indicator_protocol/{STUDY_ID}/code_sweep_and_centering.jsonl"

# The same 256 pictures per set that every boundary reading in the study used.
# fixed_image_sets seeds its own draw, so this number is the only choice here.
IMAGES_PER_SET = 256

# EuroSAT downloads about 90 MB. It is the farthest of the three picture sets
# from the training data, so it is worth having, but the notebook still works
# without it.
READ_EUROSAT = True

DATA_ROOT = Path("./data")
EUROSAT_ROOT = DATA_ROOT / "eurosat"
CACHE = Path("./cache")
CACHE.mkdir(exist_ok=True)

def secret(name):
    """Read a credential from Colab secrets, or the environment elsewhere."""
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    value = os.environ.get(name)
    if value is None:
        env = REPO / ".env"
        if env.exists():
            for line in env.read_text(encoding="utf-8").splitlines():
                if line.strip().startswith(name + "="):
                    return line.partition("=")[2].strip().strip('"').strip("'")
    return value


# R2Store holds a client, not a bucket name. `connect` builds the client from
# the four credentials and returns the store.
store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
                        secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
print("Saving readings to:", SAVE_KEY)

## 3. The pictures

The same three sets the study reads at every boundary, drawn by the same
function with the same seed, so these are the same 768 pictures.

- **in_stream**: test pictures from the nine groups the streams learn.
- **probe**: test pictures from group 9, which no stream ever sees.
- **eurosat**: satellite photographs, a different dataset altogether.

They are loaded with `ToTensor` and nothing else, which is what the study does.
No mean and standard deviation scaling is applied here. Network 4 applies its
own scaling later, on purpose, as one of the things being tested.

In [ ]:
from torch.utils.data import ConcatDataset

STREAM_TASKS = [str(i) for i in range(9)]
needed = tuple(STREAM_TASKS + [PROBE_TASK])
tasks = build_tasks(root=DATA_ROOT, partition_path=PARTITIONS[20],
                    partition_seed=42, include=needed, download=True)
print("Loaded Tiny ImageNet pictures for", len(needed), "groups")

sources = {
    "in_stream": ConcatDataset([tasks[name]["test"] for name in STREAM_TASKS]),
    "probe": tasks[PROBE_TASK]["test"],
}
if READ_EUROSAT:
    from torchvision.datasets import EuroSAT
    from torchvision.transforms import ToTensor
    sources["eurosat"] = EuroSAT(root=str(EUROSAT_ROOT), transform=ToTensor(),
                                 download=True)

image_sets = fixed_image_sets(sources, count=IMAGES_PER_SET)
for name, images in image_sets.items():
    print(f"{name:<10} {tuple(images.shape)}")

# A wrong picture size here would silently change every number below.
for name, images in image_sets.items():
    assert images.shape[1:] == (3, 64, 64), f"{name} is not 3 by 64 by 64"
    assert images.shape[0] == IMAGES_PER_SET, f"{name} has the wrong count"
print("Picture sets check out.")

## 4. The two measurements

These are copied from notebook 48 rather than imported, exactly as notebook 48
copied them from the library. The reason is the same: the centred and uncentred
numbers must go through identical arithmetic, or a difference between them
could come from the code rather than from the features.

**Effective rank** counts how many directions the features really use. It is
the exponential of the entropy of the normalised singular values, from Roy and
Vetterli 2007.

**srank** counts how many singular values are needed to reach 99 percent of
their total, from Kumar et al. 2021.

**Mean share** is the new one. Each row of the matrix is one picture's 2048
features. The average row is what every picture has in common. Mean share is
how much of the whole matrix that one row accounts for, on a scale from 0 to 1.
A value of 0.98 means the matrix is almost entirely the thing every picture
shares, and almost nothing of how the pictures differ.

In [ ]:
def rank_pair(matrix, centre):
    """Effective rank and srank of one feature matrix.

    `matrix` has one row per picture and 2048 columns. Setting `centre` to True
    subtracts the average row first, which removes what every picture has in
    common and leaves only how the pictures differ.
    """
    work = matrix.double()
    if centre:
        work = work - work.mean(dim=0, keepdim=True)
    values = torch.linalg.svdvals(work)
    total = values.sum()
    if total <= 0:
        return float("nan"), 0
    weights = values / total
    weights = weights[weights > 0]
    effective = float(torch.exp(-(weights * weights.log()).sum()).item())
    shares = torch.cumsum(values, 0) / total
    return effective, int((shares < 1 - SRANK_DELTA).sum().item()) + 1


def mean_share(matrix):
    """How much of the matrix is the one row every picture shares.

    The average row times the square root of the number of rows is the size of
    the rank one matrix made by repeating that average. Dividing by the size of
    the whole matrix gives the share, between 0 and 1.
    """
    work = matrix.double()
    norm = float(work.norm())
    if norm <= 0:
        return float("nan")
    return float(work.mean(dim=0).norm() * (work.shape[0] ** 0.5) / norm)

## 5. The four networks

Each one is built with ordinary weights. Nothing is loaded from the study and
nothing is trained.

The features are read at the same place the study reads them: the 2048 numbers
that go into the final classifier layer. Replacing that layer with `Identity`
makes the network return those numbers directly, which is simpler and less
error prone than attaching a hook.

Every network is put in evaluation mode. In training mode a BatchNorm layer
rescales using whatever batch is in front of it, which can revive a nearly dead
signal and hide the thing being measured. The study forces evaluation mode for
the same reason.

In [ ]:
def study_architecture_random(seed):
    """Network 1. The study's own target network, with ordinary weights.

    `build_target` makes the ResNet50 this study generates: a 3 by 3 stride 1
    stem instead of the stock 7 by 7 stride 2, because 64 by 64 pictures cannot
    afford the stock stem, and a spatial mean instead of the stock pool.

    In the study this network never keeps its own weights. The generator writes
    a fresh set on every forward pass. Here it keeps the weights PyTorch gave
    it at construction. That is the single difference being tested.
    """
    torch.manual_seed(seed)
    config = Config(**study_defaults("order_A"), seed=seed, device=DEVICE)
    target = build_target(config)
    target.fc = nn.Identity()
    return target.to(DEVICE).eval()


def stock_resnet50(pretrained, seed=0):
    """Networks 2 and 3. Torchvision's ResNet50, unchanged.

    `pretrained` chooses between the weights PyTorch gives a new network and the
    weights from training on ImageNet. Keeping the stock stem is the point: if
    network 1 and network 2 agree, the study's changed stem is not the cause.
    """
    if pretrained:
        model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    else:
        torch.manual_seed(seed)
        model = resnet50(weights=None)
    model.fc = nn.Identity()
    return model.to(DEVICE).eval()


# ImageNet's own scaling, used only by network 4.
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def as_imagenet_input(images):
    """Network 4's pictures: enlarged to 224 and scaled the way ImageNet is.

    A network trained at 224 with a particular scaling is being asked an unfair
    question at 64 with no scaling. This puts that right, so that a low mean
    share cannot be blamed on the input.
    """
    big = F.interpolate(images, size=224, mode="bilinear", align_corners=False)
    return (big - IMAGENET_MEAN) / IMAGENET_STD


@torch.no_grad()
def read_features(model, images, transform=None, batch_size=32):
    """The 2048 classifier inputs for one set of pictures.

    Batched because 256 pictures at 224 by 224 through a ResNet50 will not fit
    on a small GPU in one go.
    """
    collected = []
    for start in range(0, len(images), batch_size):
        batch = images[start:start + batch_size]
        if transform is not None:
            batch = transform(batch)
        collected.append(model(batch.to(DEVICE)).float().cpu())
    return torch.cat(collected)


NETWORKS = [
    ("study_arch_random", "study target, random weights, 64px",
     lambda: study_architecture_random(0), None),
    ("stock_random", "stock ResNet50, random weights, 64px",
     lambda: stock_resnet50(False), None),
    ("stock_imagenet_64", "stock ResNet50, ImageNet weights, 64px",
     lambda: stock_resnet50(True), None),
    ("stock_imagenet_224", "stock ResNet50, ImageNet weights, 224px scaled",
     lambda: stock_resnet50(True), as_imagenet_input),
]
print(f"{len(NETWORKS)} networks x {len(image_sets)} picture sets = "
      f"{len(NETWORKS) * len(image_sets)} readings")

## 6. Read them

One line printed per reading, so a network that goes wrong shows up at once
rather than at the end.

In [ ]:
import time

reads = []
for key, label, build, transform in NETWORKS:
    started = time.perf_counter()
    model = build()
    for set_name, images in image_sets.items():
        matrix = read_features(model, images, transform=transform)
        # 2048 is the width of a ResNet50's last layer. A different width here
        # means the features are being read in the wrong place.
        assert matrix.shape == (IMAGES_PER_SET, 2048), matrix.shape
        raw_e, raw_s = rank_pair(matrix, centre=False)
        mid_e, mid_s = rank_pair(matrix, centre=True)
        share = mean_share(matrix)
        reads.append(dict(
            network=key, label=label, images=set_name,
            effective_rank=raw_e, srank=raw_s,
            effective_rank_centred=mid_e, srank_centred=mid_s,
            mean_share=share, n_images=int(matrix.shape[0]),
            code_revision=HEAD, device=DEVICE))
        print(f"{key:<20}{set_name:<10} mean share {share:.4f}   "
              f"eff rank {raw_e:6.3f} -> {mid_e:7.2f}   "
              f"srank {raw_s:4d} -> {mid_s:4d}")
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"  ({time.perf_counter() - started:.0f} seconds)")
print(f"\n{len(reads)} readings done.")

## 7. The answer, next to the study's own numbers

The left side is the hypernetwork, read from notebook 48's saved file. The right
side is this notebook. The column that decides the question is mean share.

In [ ]:
import statistics
from collections import defaultdict


def mean(values):
    values = list(values)
    return sum(values) / len(values) if values else float("nan")


# ---- the study's models, from notebook 48 --------------------------------
hyper = []
if store.info(HYPERNET_KEY) is not None:
    text = store.download(HYPERNET_KEY, CACHE).read_text(encoding="utf-8")
    hyper = [json.loads(line) for line in text.splitlines() if line.strip()]
    print(f"Read {len(hyper)} hypernetwork readings from notebook 48.")
else:
    print("Notebook 48's file is not in R2 yet. The comparison is skipped.")

print()
print("MEAN SHARE: how much of the feature matrix is the one shared row")
print(f"{'network':<40}{'in_stream':>12}{'probe':>10}{'eurosat':>10}")


def row(label, rows_by_set):
    line = f"{label:<40}"
    for set_name in ("in_stream", "probe", "eurosat"):
        values = rows_by_set.get(set_name, [])
        line += f"{mean(values):>12.4f}" if values else f"{'-':>12}"
    print(line)


if hyper:
    for trained in (False, True):
        grouped = defaultdict(list)
        for r in hyper:
            if r["trained_code"] == trained:
                grouped[r["images"]].append(r["mean_share"])
        row(f"hypernetwork, {'trained' if trained else 'reference'} code", grouped)

for key, label, _, _ in NETWORKS:
    grouped = defaultdict(list)
    for r in reads:
        if r["network"] == key:
            grouped[r["images"]].append(r["mean_share"])
    row(label, grouped)

# ---- what centring does to each one --------------------------------------
print()
print("EFFECTIVE RANK, as published and after removing the shared row "
      "(in_stream pictures)")
print(f"{'network':<40}{'published':>12}{'centred':>10}{'times':>9}")
if hyper:
    for trained in (False, True):
        pub = [r["effective_rank"] for r in hyper
               if r["trained_code"] == trained and r["images"] == "in_stream"]
        cen = [r["effective_rank_centred"] for r in hyper
               if r["trained_code"] == trained and r["images"] == "in_stream"]
        if pub:
            print(f"{'hypernetwork, ' + ('trained' if trained else 'reference') + ' code':<40}"
                  f"{mean(pub):>12.3f}{mean(cen):>10.2f}{mean(cen) / mean(pub):>9.1f}")
for key, label, _, _ in NETWORKS:
    pub = [r["effective_rank"] for r in reads
           if r["network"] == key and r["images"] == "in_stream"]
    cen = [r["effective_rank_centred"] for r in reads
           if r["network"] == key and r["images"] == "in_stream"]
    if pub:
        print(f"{label:<40}{mean(pub):>12.3f}{mean(cen):>10.2f}"
              f"{mean(cen) / mean(pub):>9.1f}")

# ---- the verdict, stated plainly -----------------------------------------
print()
plain = [r["mean_share"] for r in reads if r["network"] != "study_arch_random"]
same_arch = [r["mean_share"] for r in reads if r["network"] == "study_arch_random"]
hyper_share = [r["mean_share"] for r in hyper] if hyper else []
if hyper_share:
    print(f"Hypernetwork mean share, all readings:      {mean(hyper_share):.4f}")
print(f"Study architecture, ordinary weights:       {mean(same_arch):.4f}")
print(f"Stock ResNet50, all three settings:         {mean(plain):.4f}")
print()
if mean(plain) > 0.9 and mean(same_arch) > 0.9:
    print("All of them are above 0.9. The finding is about the measure, not")
    print("about the hypernetwork, and it applies to any ReLU features.")
elif mean(same_arch) > 0.9:
    print("The architecture reads high with ordinary weights, but stock")
    print("ResNet50 does not. The cause is the changed stem or the picture")
    print("size, not the generator. Say which in the paper.")
else:
    print("Ordinary networks read much lower. The finding is about this")
    print("hypernetwork and the paper has to narrow the claim.")

## 8. Save the readings

Written to R2 so the numbers in the paper can be traced back to a file rather
than to a printed table in a notebook someone has to re-run.

In [ ]:
LOCAL = CACHE / "plain_network_centering.jsonl"
LOCAL.write_text("".join(json.dumps(r) + "\n" for r in reads), encoding="utf-8")
# Overwrite on purpose. Every reading here is a forward pass on a network built
# from a fixed seed, so a re-run reproduces the file rather than adding to it.
# Without this a second run stops with FileExistsError.
store.upload(LOCAL, SAVE_KEY, overwrite=True)
print(f"Saved {len(reads)} readings to {SAVE_KEY}")
print()
print("Paste the three tables above into docs/wiki/Experiment-trajectory-2026-11-16.md.")

## 9. What this does not settle

**One random seed per network.** The random weight networks are read once, not
across seeds. If a reading comes out near the boundary of a decision, re-run
with a different seed in `study_architecture_random` before trusting it.

**No ordinary network trained on this stream.** Network 3 is trained, but on
ImageNet, not on the nine Tiny ImageNet groups. So this notebook cannot say
what the mean share does over a continual stream in an ordinary network. It can
only say what it is for untrained and for ImageNet-trained weights. If the four
readings disagree with each other, that run becomes worth doing and it needs
real training time.

**srank is a whole number.** At 256 pictures it cannot exceed 256, so a centred
srank near 256 means the measure has run out of room rather than that the
features use exactly that many directions.